**Import e leitura de dados**

In [0]:
from pyspark.sql.functions import col, when, count, isnan, avg, round, expr, sum

df = spark.read.csv("/FileStore/tables/fashion_products.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("fashion_products")

O objetivo deste projeto é analisar dados de produtos de moda a fim de identificar padrões de vendas, categorias de marca, precificação média e nível de satisfação dos consumidores. A análise visa apoiar decisões estratégicas relacionadas ao portfólio de produtos, marketing e relacionamento com marcas.

**Qualidade dos dados**

In [0]:
# Valores nulos por coluna
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Valores negativos de preço
df.filter(col("price") < 0).show()

# Valores fora do intervalo de rating
df.filter((col("rating") < 1) | (col("rating") > 5)).show()

+-------+----------+------------+-----+--------+-----+------+-----+----+
|User ID|Product ID|Product Name|Brand|Category|Price|Rating|Color|Size|
+-------+----------+------------+-----+--------+-----+------+-----+----+
|      0|         0|           0|    0|       0|    0|     0|    0|   0|
+-------+----------+------------+-----+--------+-----+------+-----+----+

+-------+----------+------------+-----+--------+-----+------+-----+----+
|User ID|Product ID|Product Name|Brand|Category|Price|Rating|Color|Size|
+-------+----------+------------+-----+--------+-----+------+-----+----+
+-------+----------+------------+-----+--------+-----+------+-----+----+

+-------+----------+------------+-----+--------+-----+------+-----+----+
|User ID|Product ID|Product Name|Brand|Category|Price|Rating|Color|Size|
+-------+----------+------------+-----+--------+-----+------+-----+----+
+-------+----------+------------+-----+--------+-----+------+-----+----+



**Renomeação das colunas**

In [0]:
df = df.withColumnRenamed("Product Name", "product_name") \
       .withColumnRenamed("Price", "price") \
       .withColumnRenamed("Brand", "brand") \
       .withColumnRenamed("Rating", "rating") \
       .withColumnRenamed("User ID", "user_id") \
       .withColumnRenamed("Product ID", "product_id") \
       .withColumnRenamed("Color", "color") \
       .withColumnRenamed("Size", "size") \
       .withColumnRenamed("Category", "category")


**Enriquecimento dos dados**

In [0]:
# Agrupamento de marcas por grupo
df = df.withColumn("grupo_marca", when(col("brand").isin("Nike", "Adidas"), "Sport")
                                    .when(col("brand").isin("Zara", "H&M"), "Fast Fashion")
                                    .when(col("brand").isin("Gucci"), "Slow Fashion")
                                    .otherwise("Outros"))

# Cálculo da média de satisfação por marca
df_satisfacao = df.groupBy("brand").agg(avg("rating").alias("indice_satisfacao"))

# Classificação em fases
fase = when(col("indice_satisfacao") >= 4.5, "Muito bom") \
       .when(col("indice_satisfacao") >= 4.0, "Regular") \
       .otherwise("Ruim")

df_satisfacao = df_satisfacao.withColumn("fase", fase)
df = df.join(df_satisfacao, on="brand", how="left")
df.createOrReplaceTempView("fashion_products")

**Bronze Layer**

In [0]:
raw_df = spark.read.csv("/FileStore/tables/fashion_products.csv", header=True, inferSchema=True)

for c in raw_df.columns:
    new_name = c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    raw_df = raw_df.withColumnRenamed(c, new_name)
raw_df.printSchema()

raw_df.write.format("delta").mode("overwrite").save("/mnt/lake/bronze/fashion_products")

root
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fashion_bronze;

CREATE TABLE IF NOT EXISTS fashion_bronze.fashion_products
USING DELTA
LOCATION '/mnt/lake/bronze/fashion_products';


In [0]:
%sql
SELECT * FROM fashion_bronze.fashion_products LIMIT 10;

user_id,product_id,product_name,brand,category,price,rating,color,size
19,1,Dress,Adidas,Men's Fashion,40,1.0431592108361825,Black,XL
97,2,Shoes,H&M,Women's Fashion,82,4.026416271141911,Black,L
25,3,Dress,Adidas,Women's Fashion,44,3.337937559377053,Yellow,XL
57,4,Shoes,Zara,Men's Fashion,23,1.0495229563128543,White,S
79,5,T-shirt,Adidas,Men's Fashion,79,4.302773408398684,Black,M
98,6,Dress,Adidas,Men's Fashion,47,1.3795657395330458,Yellow,L
16,7,Jeans,Gucci,Men's Fashion,37,1.3567503746842564,White,XL
63,8,Sweater,Zara,Kids' Fashion,64,4.36030328941572,Blue,XL
96,9,Sweater,H&M,Men's Fashion,53,4.466181876278437,Green,XL
36,10,T-shirt,Zara,Kids' Fashion,55,4.093234402033421,White,XL


**Silver Layer**

Na camada Silver, foram realizados os seguintes tratamentos: Conversão de tipos (price e rating para double), Criação da coluna grupo_marca com base na marca (ex: Sport, Fast Fashion, Slow Fashion) e	Remoção de registros com valores nulos ou inválidos. Também foi incluido na ETL a Conversão de tipos,	Criação da coluna grupo_marca com when().otherwise() e Filtro de registros nulos



In [0]:
df = spark.read.format("delta").load("/mnt/lake/bronze/fashion_products")

df = df.filter(df.price > 0)
df = df.withColumn("grupo_marca", when(df.brand.isin("Nike", "Adidas"), "Esportiva")
                                  .when(df.brand.isin("Zara", "H&M"), "Fast Fashion")
                                  .when(df.brand.isin("Gucci"), "Slow Fashion")
                                  .otherwise("Outros"))

df.write.format("delta").mode("overwrite").save("/mnt/lake/silver/fashion_products")


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fashion_silver;

CREATE TABLE IF NOT EXISTS fashion_silver.fashion_products
USING DELTA
LOCATION '/mnt/lake/silver/fashion_products';


In [0]:
%sql
SELECT * FROM fashion_silver.fashion_products LIMIT 10;


user_id,product_id,product_name,brand,category,price,rating,color,size,grupo_marca
19,1,Dress,Adidas,Men's Fashion,40,1.0431592108361825,Black,XL,Esportiva
97,2,Shoes,H&M,Women's Fashion,82,4.026416271141911,Black,L,Fast Fashion
25,3,Dress,Adidas,Women's Fashion,44,3.337937559377053,Yellow,XL,Esportiva
57,4,Shoes,Zara,Men's Fashion,23,1.0495229563128543,White,S,Fast Fashion
79,5,T-shirt,Adidas,Men's Fashion,79,4.302773408398684,Black,M,Esportiva
98,6,Dress,Adidas,Men's Fashion,47,1.3795657395330458,Yellow,L,Esportiva
16,7,Jeans,Gucci,Men's Fashion,37,1.3567503746842564,White,XL,Slow Fashion
63,8,Sweater,Zara,Kids' Fashion,64,4.36030328941572,Blue,XL,Fast Fashion
96,9,Sweater,H&M,Men's Fashion,53,4.466181876278437,Green,XL,Fast Fashion
36,10,T-shirt,Zara,Kids' Fashion,55,4.093234402033421,White,XL,Fast Fashion


**Golden layer**

As principais métricas geradas na Gold Layer incluem: preco_medio: preço médio por marca e grupo de marca, indice_satisfacao: média do rating por marca e grupo e total_vendas: soma de preços por produto (como proxy de volume de vendas). Na ETL foi feito o Agrupamentos com groupBy, Criação de métricas com avg() e sum() e Escrita das tabelas finais no Data Lake (pasta /mnt/lake/gold/fashion_insights)



In [0]:
df = df.withColumn("price", col("price").cast("double")) \
       .withColumn("rating", col("rating").cast("double"))

df_agg = df.groupBy("brand", "grupo_marca").agg(
    avg("rating").alias("indice_satisfacao"),
    avg("price").alias("preco_medio"),
    sum("price").alias("total_vendas")
)

df_agg.write.format("delta").mode("overwrite").save("/mnt/lake/gold/fashion_products")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fashion_gold;

CREATE TABLE IF NOT EXISTS fashion_gold.fashion_products
USING DELTA
LOCATION '/mnt/lake/gold/fashion_products';


In [0]:
%sql
SELECT * FROM fashion_gold.fashion_products LIMIT 10;


brand,grupo_marca,indice_satisfacao,preco_medio,total_vendas
Gucci,Slow Fashion,3.162277299075944,55.42931937172775,10587.0
Zara,Fast Fashion,3.0015937093534126,54.748768472906406,11114.0
H&M,Fast Fashion,2.99630915039578,57.56701030927835,11168.0
Adidas,Esportiva,2.942719518488803,53.515151515151516,10596.0
Nike,Esportiva,2.877916723375101,57.570093457943926,12320.0


•	Quais são os produtos mais vendidos?

In [0]:
%sql
SELECT brand, SUM(Price) AS total_vendas
FROM fashion_products
GROUP BY brand
ORDER BY total_vendas DESC;

brand,total_vendas
Nike,12320
H&M,11168
Zara,11114
Adidas,10596
Gucci,10587


•	Qual a distribuição dos preços por categoria?

In [0]:
%sql
SELECT category, MIN(price), MAX(price), AVG(price), PERCENTILE(price, 0.5) AS mediana
FROM fashion_products
GROUP BY category;

category,min(price),max(price),avg(price),mediana
Kids' Fashion,10,100,56.51282051282051,59.0
Men's Fashion,10,100,53.67080745341615,51.5
Women's Fashion,10,99,57.08562691131498,58.0


•	Qual é o preço total dos produtos?

In [0]:
%sql
SELECT 
  product_name, 
  ROUND(SUM(price), 2) AS total_vendido
FROM fashion_products
GROUP BY product_name
ORDER BY total_vendido DESC
LIMIT 10;

product_name,total_vendido
Jeans,13097
Shoes,12596
T-shirt,11399
Dress,9379
Sweater,9314


•	Qual é o preço médio dos produtos?

In [0]:
%sql
SELECT 
  product_name AS nome_produto, 
  ROUND(AVG(price), 2) AS preco_medio
FROM fashion_products
GROUP BY product_name
ORDER BY preco_medio DESC
LIMIT 10;


nome_produto,preco_medio
Shoes,56.74
T-shirt,56.71
Jeans,56.7
Sweater,54.79
Dress,53.29


•	Como está o índice de satisfação (rating) dos clientes por marca?

In [0]:
%sql
SELECT brand, ROUND(AVG(rating),2) AS indice_satisfacao
FROM fashion_products
GROUP BY brand
ORDER BY indice_satisfacao DESC;

brand,indice_satisfacao
Gucci,3.16
Zara,3.0
H&M,3.0
Adidas,2.94
Nike,2.88


•	Qual o total de vendas por grupos como esportiva, fast fashion ou slow fashion?

In [0]:
%sql
SELECT grupo_marca, SUM(Price) AS total_vendas
FROM fashion_products
GROUP BY grupo_marca
ORDER BY total_vendas DESC;

grupo_marca,total_vendas
Sport,22916
Fast Fashion,22282
Slow Fashion,10587
